In [2]:
%pip install pytest

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
reference_data = [
  {
    "question": "What is the Privacy Policy?", 
    "ground_truth": "This Privacy Policy explains how Yellow Sapphire Consulting, collects, uses, and protects your personal information when you use the SpeakEQ mobile app.We comply with privacy laws including GDPR (for European users) and CCPA (for California users).", #Expected llm generated answer
    "context": "This Privacy Policy explains how Yellow Sapphire Consulting, collects, uses, and protects your personal information when you use the SpeakEQ mobile app.We comply with privacy laws including GDPR (for European users) and CCPA (for California users)." #Expected retrieved context
  }
]

# reference_data = [
#   {
#     "question": "What is the Privacy Policy?", 
#     "ground_truth": "This Privacy Policy explains how Yellow Sapphire Consulting, collects, uses, and protects your personal information when you use the SpeakEQ mobile app.We comply with privacy laws including GDPR (for European users) and CCPA (for California users)." #Expected llm generated answer
#     "context": "This Privacy Policy explains how Yellow Sapphire Consulting, collects, uses, and protects your personal information when you use the SpeakEQ mobile app.We comply with privacy laws including GDPR (for European users) and CCPA (for California users)." #Expected retrieved context
#   }
# ]
question = reference_data[0]['question']
ground_truth = reference_data[0]['ground_truth']
context = reference_data[0]['context']
print (f"question: {question}")
print (f"ground_truth: {ground_truth}")
print (f"context: {context}")

question: What is the Privacy Policy?
ground_truth: This Privacy Policy explains how Yellow Sapphire Consulting, collects, uses, and protects your personal information when you use the SpeakEQ mobile app.We comply with privacy laws including GDPR (for European users) and CCPA (for California users).
context: This Privacy Policy explains how Yellow Sapphire Consulting, collects, uses, and protects your personal information when you use the SpeakEQ mobile app.We comply with privacy laws including GDPR (for European users) and CCPA (for California users).


In [4]:
# Retrieve context from Milvus DB

from milvus_chatbot_with_rag import retrieve_similiar_contexts, generate_answer

def perform_retrieval(question):

    retrieved_context = retrieve_similiar_contexts(question, "policy_docs_collection", 1)[0]['content']
    print (f"perform_retrieval.retrieved_context: {retrieved_context}")
    return retrieved_context

# Generate answer using LLM

question = reference_data[0]['question']
context = perform_retrieval(question)
answer = generate_answer(question, context)
answer

Connected to Milvus on Zilliz Cloud
perform_retrieval.retrieved_context: Privacy Policy On the SpeakEQ app, we respect your privacy. This Privacy Policy explains how Yellow Sapphire Consulting, LLC DBA SpeakEQ ("we," "us," or "our") collects, uses, and protects your personal information when you use the SpeakEQ mobile app. We comply with privacy laws including GDPR (for European users) and CCPA (for California users).


'The Privacy Policy states that the SpeakEQ app respects your privacy. It explains how Yellow Sapphire Consulting, LLC (SpeakEQ, “we,” “us,” or “our”) collects, uses, and protects your personal information when you use the SpeakEQ mobile app. It also notes that we comply with privacy laws including GDPR (for European users) and CCPA (for California users).'

In [5]:
%pip install ragas datasets 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_correctness

from dotenv import load_dotenv
from openai import OpenAI
import os

# --- Load API Key ---
load_dotenv(override=True, dotenv_path="../.env")
my_api_key = os.getenv("OPENAI_API_KEY")


client = OpenAI(api_key=my_api_key)

# Question User asked
question = reference_data[0]['question']

# Reference context (should be a string)
reference_context = reference_data[0]['context']

# ground truth answer
ground_truth = reference_data[0]['ground_truth']

# Retrieved context (a string from perform_retrieval)
retrieved_context = [perform_retrieval(question)]
llm_answer = generate_answer(question, retrieved_context[0])

# Build dataset properly
dataset_dict = {
    "question": [question],
    "contexts": [retrieved_context],    # list of strings INSIDE another list
    "ground_truth": [ground_truth],   # single string/ reference answer
    "answer": [llm_answer]
}

print(f"dataset_dict: {dataset_dict}")

ragas_dataset = Dataset.from_dict(dataset_dict)

C:\Users\nirma\AppData\Local\Temp\ipykernel_53488\3667568502.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_correctness
C:\Users\nirma\AppData\Local\Temp\ipykernel_53488\3667568502.py:3: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import faithfulness, answer_correctness


Connected to Milvus on Zilliz Cloud
perform_retrieval.retrieved_context: Privacy Policy On the SpeakEQ app, we respect your privacy. This Privacy Policy explains how Yellow Sapphire Consulting, LLC DBA SpeakEQ ("we," "us," or "our") collects, uses, and protects your personal information when you use the SpeakEQ mobile app. We comply with privacy laws including GDPR (for European users) and CCPA (for California users).
dataset_dict: {'question': ['What is the Privacy Policy?'], 'contexts': [['Privacy Policy On the SpeakEQ app, we respect your privacy. This Privacy Policy explains how Yellow Sapphire Consulting, LLC DBA SpeakEQ ("we," "us," or "our") collects, uses, and protects your personal information when you use the SpeakEQ mobile app. We comply with privacy laws including GDPR (for European users) and CCPA (for California users).']], 'ground_truth': ['This Privacy Policy explains how Yellow Sapphire Consulting, collects, uses, and protects your personal information when you use the

In [7]:
from ragas.llms.base import llm_factory
from ragas import evaluate
from ragas.metrics import answer_correctness

results = evaluate(
    dataset=ragas_dataset,
    metrics=[faithfulness, answer_correctness]  
)


print("LLM Generation Evaluation Results:")
results.to_pandas()

C:\Users\nirma\AppData\Local\Temp\ipykernel_53488\1314932040.py:3: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import answer_correctness


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

Exception raised in Job[1]: TypeError(Cannot use aembed_text() with a synchronous client. Use embed_text() instead.)


LLM Generation Evaluation Results:


,user_input,retrieved_contexts,response,reference,faithfulness,answer_correctness
0,What is the Privacy Policy?,"[Privacy Policy On the SpeakEQ app, we respect...",The Privacy Policy is a document from Yellow S...,This Privacy Policy explains how Yellow Sapphi...,1.0,NaN


In [8]:
from ragas.llms.base import llm_factory
from ragas import evaluate
from ragas.metrics import answer_correctness

# Create the modern LLM wrapper
client = OpenAI()
llm = llm_factory("gpt-4o-mini", client=client)

# Run evaluation
results = evaluate(
    dataset=ragas_dataset,
    metrics=[answer_correctness],
    llm=llm
)

print("LLM Generation Evaluation Results:")
results.to_pandas()

C:\Users\nirma\AppData\Local\Temp\ipykernel_53488\1016587815.py:3: DeprecationWarning: Importing answer_correctness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_correctness
  from ragas.metrics import answer_correctness


Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

Exception raised in Job[0]: TypeError(Cannot use aembed_text() with a synchronous client. Use embed_text() instead.)


LLM Generation Evaluation Results:


,user_input,retrieved_contexts,response,reference,answer_correctness
0,What is the Privacy Policy?,"[Privacy Policy On the SpeakEQ app, we respect...",The Privacy Policy is a document from Yellow S...,This Privacy Policy explains how Yellow Sapphi...,NaN
